# 03 - Mainnet CLMM

Concentrated-liquidity AMM analysis. This notebook is intentionally separate from CPMM because CLMM needs tick/active-liquidity state and cannot reuse the CPMM closed-form baseline.

Expected future input:
- `results/historical_clmm_candidates.csv`


## Load data


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "helpers").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "helpers").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

from helpers import (
    best_conditions,
    blocker_table,
    data_readiness,
    find_repo_root,
    historical_counterfactual_summary,
    hypothesis_scorecard,
    load_inputs,
    plot_realized_heatmap,
    plot_sensitivity_lines,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)

ROOT = find_repo_root()
inputs = load_inputs(ROOT)

clmm = inputs["frames"]["historical_clmm"]
cpmm = inputs["frames"]["historical_cpmm"]

display(data_readiness(ROOT, inputs, ["historical_clmm", "historical_cpmm"]))


## Scope check


In [ ]:
if clmm.empty:
    display(Markdown(
        "No CLMM candidate CSV exists yet. This notebook is a contract for Task E, not final evidence. "
        "Needed next: choose CLMM pool/window, decode swaps, reconstruct tick arrays/pre-state, run a CLMM numerical sandwich simulator."
    ))
else:
    display(historical_counterfactual_summary(clmm))


## Required CLMM candidate fields


In [ ]:
required = pd.DataFrame([
    {"field": "pool_type", "why": "distinguish raydium_clmm/orca_whirlpool from cpmm"},
    {"field": "pool_label, pool_address", "why": "group results by selected pool"},
    {"field": "slot, signature, instruction_index", "why": "trace every counterfactual row back to a historical swap"},
    {"field": "amount_in, min_amount_out, actual_amount_out", "why": "victim size and slippage bound"},
    {"field": "sqrt_price_x64_before, liquidity_before, tick_current_before", "why": "CLMM state at victim pre-state"},
    {"field": "tick_arrays_before", "why": "needed for executable CLMM swap simulation across ticks"},
    {"field": "fee_rate, protocol_fee_rate", "why": "fee-aware profitability"},
    {"field": "tx_cost_per_leg", "why": "net profit, not just gross extraction"},
])
display(required)


## CPMM vs CLMM comparison


In [ ]:
if clmm.empty or cpmm.empty:
    display(Markdown("Comparison waits for both historical CPMM and CLMM candidate outputs."))
else:
    cpmm_summary = historical_counterfactual_summary(cpmm).assign(family="CPMM")
    clmm_summary = historical_counterfactual_summary(clmm).assign(family="CLMM")
    display(pd.concat([cpmm_summary, clmm_summary], ignore_index=True))


## Thesis-ready takeaways


In [ ]:
if clmm.empty:
    display(Markdown(
        "- CLMM remains an empirical gap, not a result.\n"
        "- Keep this notebook because it defines the expected CLMM evidence path and prevents mixing CPMM assumptions into CLMM analysis."
    ))
else:
    display(Markdown(f"- Historical CLMM candidates loaded: `{len(clmm)}` rows."))
